# Notebook 01 — Atoms, Molecules & Constants

**MOLEKUL | Phase 1**

---

Every quantum chemistry calculation starts with the same question: *what are we computing?*  
The answer is a molecule — a collection of nuclei and electrons frozen (for now) at specified positions.

In this notebook we build the two fundamental data structures that every later module depends on:

- `Atom` — a nucleus with a symbol, an atomic number, and a position in 3D space.
- `Molecule` — a list of atoms plus charge and spin multiplicity.

We also introduce the unit system MOLEKUL uses internally (atomic units) and the conversion
constants that let us move between atomic units and the SI-derived units most chemists prefer.

**What you will learn:**
1. Why atomic units exist and what they mean physically.
2. How to create atoms from Angstrom coordinates.
3. How to assemble molecules and compute the nuclear repulsion energy analytically.
4. How charge and spin multiplicity constrain the electron count.

## 1. Atomic units

In SI units, every quantum chemistry formula is cluttered with $\hbar$, $m_e$, $e$, and $4\pi\varepsilon_0$.
Atomic units eliminate that clutter by defining:

| Quantity | Symbol | Value in SI |
|----------|--------|-------------|
| Length (Bohr) | $a_0$ | $5.2918 \times 10^{-11}$ m |
| Energy (Hartree) | $E_h$ | $4.3597 \times 10^{-18}$ J |
| Mass | $m_e$ | $9.1094 \times 10^{-31}$ kg |
| Charge | $e$ | $1.6022 \times 10^{-19}$ C |

With these choices $\hbar = m_e = e = 4\pi\varepsilon_0 = 1$ and the Schrödinger equation
for the hydrogen atom becomes $\hat{H} = -\tfrac{1}{2}\nabla^2 - 1/r$.

**Rule:** MOLEKUL stores *all* coordinates in **Bohr** and all energies in **Hartree**.
The `constants.py` module provides the conversion factors you need to go back to human-readable units.

In [1]:
# --- Make the MOLEKUL package importable -------------------------------
# Best practice: install once from the repo root with
#     pip install -e ".[notebooks]"
# The fallback below locates the in-repo src/ automatically, so the
# notebook also runs from a fresh clone that has not been installed yet,
# regardless of which directory Jupyter was started from.
try:
    import molekul  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        if (_p / "src" / "molekul").is_dir():
            sys.path.insert(0, str(_p / "src"))
            break
    import molekul  # noqa: F401
# -----------------------------------------------------------------------

import numpy as np
from molekul.constants import (
    ANGSTROM_TO_BOHR,
    BOHR_TO_ANGSTROM,
    HARTREE_TO_EV,
    HARTREE_TO_KCAL_MOL,
)

print(f"1 Angstrom = {ANGSTROM_TO_BOHR:.6f} Bohr")
print(f"1 Hartree  = {HARTREE_TO_EV:.6f} eV")
print(f"1 Hartree  = {HARTREE_TO_KCAL_MOL:.4f} kcal/mol")

1 Angstrom = 1.889726 Bohr
1 Hartree  = 27.211396 eV
1 Hartree  = 627.5095 kcal/mol


## 2. The `Atom` class

An `Atom` is a dataclass with three fields:
- `symbol` — element symbol string, e.g. `"O"`.
- `coords` — NumPy array of shape `(3,)` in **Bohr**.
- `Z` (property) — atomic number, looked up from `SYMBOL_TO_Z`.

The canonical constructor takes coords in Bohr directly.  
The class method `Atom.from_angstrom()` is the convenient entry point when you have
crystallographic or optimised geometry data in Angstrom.

In [2]:
from molekul.atoms import Atom

# Create hydrogen at origin — coords in Bohr
h1 = Atom(symbol="H", coords=np.array([0.0, 0.0, 0.0]))
print(h1)
print(f"  Z = {h1.Z}, coords (Bohr) = {h1.coords}")

# Create oxygen from Angstrom coordinates
o = Atom.from_angstrom("O", 0.0, 0.0, 0.1173)
print()
print(o)
print(f"  coords (Angstrom) = {o.coords_angstrom()}")
print(f"  coords (Bohr)     = {o.coords}")

Atom(H, Z=1, xyz=[0.000000, 0.000000, 0.000000] Å)
  Z = 1, coords (Bohr) = [0. 0. 0.]

Atom(O, Z=8, xyz=[0.000000, 0.000000, 0.117300] Å)
  coords (Angstrom) = [0.     0.     0.1173]
  coords (Bohr)     = [0.         0.         0.22166486]


### What happens inside `Atom.__post_init__`?

Python dataclasses call `__post_init__` immediately after `__init__`.  
MOLEKUL uses it to:
1. Validate that `symbol` is in the periodic table lookup table.
2. Cast `coords` to a `float64` NumPy array.
3. Confirm the shape is `(3,)`.

This means any typo in the element symbol is caught at creation time, not buried
in an obscure integral error 500 lines later.

In [3]:
# Unknown element raises immediately
try:
    bad = Atom(symbol="Xx", coords=np.zeros(3))
except ValueError as e:
    print(f"ValueError: {e}")

# Wrong shape raises immediately
try:
    bad2 = Atom(symbol="H", coords=np.zeros(2))
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: Unknown element symbol: 'Xx'
ValueError: coords must be shape (3,), got (2,)


## 3. The `Molecule` class

A `Molecule` wraps a list of `Atom` objects and adds:
- `charge` — total charge in units of $e$ (integer).
- `multiplicity` — spin multiplicity $2S+1$ (integer, ≥ 1).
- `name` — optional label string.

The number of electrons is determined automatically:
$$N_e = \sum_A Z_A - \text{charge}$$

The alpha/beta electron split follows from the multiplicity:
$$N_\alpha = \frac{N_e + (2S)}{2}, \quad N_\beta = N_e - N_\alpha$$

`__post_init__` checks consistency: you cannot have an odd number of paired electrons.

In [4]:
from molekul.molecule import Molecule

# Water in its experimental geometry (Angstrom)
# O at origin, H atoms symmetrically placed
h2o = Molecule(
    atoms=[
        Atom.from_angstrom("O",  0.0000,  0.0000,  0.1173),
        Atom.from_angstrom("H",  0.7572,  0.0000, -0.4692),
        Atom.from_angstrom("H", -0.7572,  0.0000, -0.4692),
    ],
    charge=0,
    multiplicity=1,
    name="water",
)
print(h2o)

Molecule(name='water', charge=0, mult=1)
  Atom(O, Z=8, xyz=[0.000000, 0.000000, 0.117300] Å)
  Atom(H, Z=1, xyz=[0.757200, 0.000000, -0.469200] Å)
  Atom(H, Z=1, xyz=[-0.757200, 0.000000, -0.469200] Å)
  n_electrons=10, n_alpha=5, n_beta=5


In [5]:
print(f"Atoms:        {h2o.n_atoms}")
print(f"Electrons:    {h2o.n_electrons}  (8 from O + 2*1 from H − charge 0)")
print(f"Alpha/Beta:   {h2o.n_alpha} / {h2o.n_beta}")

# Coordinate matrix
print(f"\nCoords (Bohr), shape {h2o.coords_bohr.shape}:")
print(h2o.coords_bohr)

Atoms:        3
Electrons:    10  (8 from O + 2*1 from H − charge 0)
Alpha/Beta:   5 / 5

Coords (Bohr), shape (3, 3):
[[ 0.          0.          0.22166486]
 [ 1.43090052  0.         -0.88665943]
 [-1.43090052  0.         -0.88665943]]


## 4. Nuclear repulsion energy

The simplest quantity we can compute from atomic positions alone is the **nuclear repulsion energy**:

$$E_{\text{nuc}} = \sum_{A < B} \frac{Z_A Z_B}{|\mathbf{R}_A - \mathbf{R}_B|}$$

This is a purely classical electrostatic term — point charges interacting via Coulomb's law.
In the Born-Oppenheimer approximation the nuclei are fixed, so $E_{\text{nuc}}$ is just a
constant added to the electronic energy at the end.

The formula is exact (no approximation) and cheap to compute ($\mathcal{O}(N^2)$ in atoms).

In [6]:
E_nuc = h2o.nuclear_repulsion_energy()
print(f"E_nuc(H2O) = {E_nuc:.6f} Hartree")
print(f"           = {E_nuc * HARTREE_TO_EV:.4f} eV")
print(f"           = {E_nuc * HARTREE_TO_KCAL_MOL:.2f} kcal/mol")

# Reference: PySCF gives 9.189533 Ha for this geometry
ref = 9.189533
print(f"\nDiff from PySCF reference: {abs(E_nuc - ref):.2e} Ha")

E_nuc(H2O) = 9.189534 Hartree
           = 250.0601 eV
           = 5766.52 kcal/mol

Diff from PySCF reference: 1.42e-06 Ha


## 5. More molecules

Let us build a few more systems to get comfortable with the API.

In [7]:
# H2 at experimental bond length 0.74 Angstrom
h2 = Molecule(
    atoms=[
        Atom.from_angstrom("H", 0.0, 0.0,  0.37),
        Atom.from_angstrom("H", 0.0, 0.0, -0.37),
    ],
    name="H2",
)
print(h2)
print(f"E_nuc = {h2.nuclear_repulsion_energy():.6f} Ha")

Molecule(name='H2', charge=0, mult=1)
  Atom(H, Z=1, xyz=[0.000000, 0.000000, 0.370000] Å)
  Atom(H, Z=1, xyz=[0.000000, 0.000000, -0.370000] Å)
  n_electrons=2, n_alpha=1, n_beta=1
E_nuc = 0.715104 Ha


In [8]:
# HeH+ — smallest stable molecule, charge = +1
heh = Molecule(
    atoms=[
        Atom.from_angstrom("He", 0.0, 0.0, 0.0),
        Atom.from_angstrom("H",  0.0, 0.0, 0.7743),
    ],
    charge=1,
    name="HeH+",
)
print(heh)
print(f"E_nuc = {heh.nuclear_repulsion_energy():.6f} Ha")

Molecule(name='HeH+', charge=1, mult=1)
  Atom(He, Z=2, xyz=[0.000000, 0.000000, 0.000000] Å)
  Atom(H, Z=1, xyz=[0.000000, 0.000000, 0.774300] Å)
  n_electrons=2, n_alpha=1, n_beta=1
E_nuc = 1.366853 Ha


In [9]:
# OH radical — open-shell doublet (multiplicity = 2)
oh = Molecule(
    atoms=[
        Atom.from_angstrom("O", 0.0, 0.0, 0.0),
        Atom.from_angstrom("H", 0.0, 0.0, 0.9697),
    ],
    multiplicity=2,
    name="OH radical",
)
print(oh)
print(f"n_alpha={oh.n_alpha}, n_beta={oh.n_beta}  (1 unpaired electron)")

Molecule(name='OH radical', charge=0, mult=2)
  Atom(O, Z=8, xyz=[0.000000, 0.000000, 0.000000] Å)
  Atom(H, Z=1, xyz=[0.000000, 0.000000, 0.969700] Å)
  n_electrons=9, n_alpha=5, n_beta=4
n_alpha=5, n_beta=4  (1 unpaired electron)


## 6. Reading XYZ files

MOLEKUL provides `io_xyz.py` to read the `.xyz` format, the simplest geometry file format
used across computational chemistry:

```
3
water
O  0.0000  0.0000  0.1173
H  0.7572  0.0000 -0.4692
H -0.7572  0.0000 -0.4692
```

The first line is the atom count, the second is a comment, and the remaining lines list
element symbol and Cartesian coordinates in Angstrom.

In [10]:
import tempfile, os
from molekul.io_xyz import read_xyz

xyz_text = """3
water molecule
O  0.0000  0.0000  0.1173
H  0.7572  0.0000 -0.4692
H -0.7572  0.0000 -0.4692
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".xyz", delete=False) as f:
    f.write(xyz_text)
    tmp = f.name

mol_from_file = read_xyz(tmp)
os.unlink(tmp)

print(mol_from_file)
print(f"E_nuc = {mol_from_file.nuclear_repulsion_energy():.6f} Ha")

Molecule(name='water molecule', charge=0, mult=1)
  Atom(O, Z=8, xyz=[0.000000, 0.000000, 0.117300] Å)
  Atom(H, Z=1, xyz=[0.757200, 0.000000, -0.469200] Å)
  Atom(H, Z=1, xyz=[-0.757200, 0.000000, -0.469200] Å)
  n_electrons=10, n_alpha=5, n_beta=5
E_nuc = 9.189534 Ha


## 7. Element coverage — what you can actually compute

It helps to separate two different limits.

* **Building an `Atom` / `Molecule`.** The symbol → Z table in `constants.py`
  knows the first 18 elements (H–Ar), so you can *construct* anything up to argon.
* **Running a calculation.** A calculation also needs (a) a basis set for every
  element present and (b), for vibrational analysis, an atomic mass. Those tables
  are smaller, so the genuinely *usable* range is narrower:

| Capability | Elements | Z |
|------------|----------|----|
| `Atom` / `Molecule` objects | H – Ar | 1–18 |
| STO-3G basis | H – Ne | 1–10 |
| 6-31G\* and cc-pVDZ bases | H, He, C, N, O, F | — |
| Atomic masses (frequencies, phonons) | H – Ne | 1–10 |

So you *can* build SiH₄, but asking for its RHF/cc-pVDZ energy will raise an error,
because cc-pVDZ has no silicon shell. The worked examples in this series —
H₂, H₂O, N₂, CH₄, LiH — were chosen to sit inside *every* one of these tables.

In [11]:
from molekul.constants import Z_TO_SYMBOL, ATOMIC_MASS
from molekul.basis_sto3g import STO3G
from molekul.basis_631gstar import G631Star
from molekul.basis_ccpvdz import ccpVDZ

# Which elements does each table actually cover?
sto    = set(STO3G.shells_by_element)
g631   = set(G631Star.shells_by_element)
ccpvdz = set(ccpVDZ.shells_by_element)

def mark(ok):
    return "yes" if ok else " - "

print(f"{'Z':>3}  {'sym':>3} | {'STO-3G':^7}|{'6-31G*':^7}|{'cc-pVDZ':^7}| {'mass':^5}")
print("-" * 47)
for z, sym in sorted(Z_TO_SYMBOL.items()):
    print(f"{z:>3}  {sym:>3} |  {mark(sym in sto)}  |  {mark(sym in g631)}  "
          f"|  {mark(sym in ccpvdz)}  |  {mark(z in ATOMIC_MASS)}")

print("\\nAtom objects span H-Ar; real calculations are limited by the basis/mass tables above.")

  Z  sym | STO-3G |6-31G* |cc-pVDZ| mass 
-----------------------------------------------
  1    H |  yes  |  yes  |  yes  |  yes
  2   He |  yes  |  yes  |  yes  |  yes
  3   Li |  yes  |   -   |   -   |  yes
  4   Be |  yes  |   -   |   -   |  yes
  5    B |  yes  |   -   |   -   |  yes
  6    C |  yes  |  yes  |  yes  |  yes
  7    N |  yes  |  yes  |  yes  |  yes
  8    O |  yes  |  yes  |  yes  |  yes
  9    F |  yes  |  yes  |  yes  |  yes
 10   Ne |  yes  |   -   |   -   |  yes
 11   Na |   -   |   -   |   -   |   - 
 12   Mg |   -   |   -   |   -   |   - 
 13   Al |   -   |   -   |   -   |   - 
 14   Si |   -   |   -   |   -   |   - 
 15    P |   -   |   -   |   -   |   - 
 16    S |   -   |   -   |   -   |   - 
 17   Cl |   -   |   -   |   -   |   - 
 18   Ar |   -   |   -   |   -   |   - 
\nAtom objects span H-Ar; real calculations are limited by the basis/mass tables above.


---

## Exercises

**1.** Build a nitrogen molecule (N₂) at the experimental bond length of 1.098 Å and compute its
nuclear repulsion energy. What does the $Z_A Z_B$ factor tell you about why this number is
large compared to H₂?

**2.** Create a water cation (H₂O⁺, charge = +1). What is the multiplicity of the ground state
(one electron has been removed from a doubly-occupied orbital)?  
Confirm that `Molecule` accepts your answer.

**3.** Try to create a molecule with an inconsistent charge/multiplicity combination
(e.g., H₂ with charge 0 and multiplicity 2). What error does MOLEKUL raise, and why?

**4.** Write a function that takes a `Molecule` and prints a summary table:
   atom index, symbol, Z, and (x, y, z) in Angstrom — one row per atom.

**5.** (Advanced) The nuclear repulsion energy of a homonuclear diatomic $X_2$ scales as
$Z^2 / R$. For H₂, N₂, and F₂ at their experimental bond lengths (0.74, 1.098, 1.418 Å),
compute $E_{\text{nuc}}$ and verify this scaling numerically.

---

## Summary

| Concept | Key point |
|---------|----------|
| Atomic units | Bohr (length), Hartree (energy); eliminates $\hbar$, $m_e$, $e$ from equations |
| `Atom` | Symbol + 3D position in Bohr; Z looked up automatically |
| `Molecule` | List of atoms + charge + multiplicity; validates consistency |
| $E_{\text{nuc}}$ | Classical Coulomb sum; exact, $\mathcal{O}(N^2)$, added at the end |
| XYZ format | Coords in Angstrom; `read_xyz()` converts internally |

In the next notebook we compute the one-electron integrals — the overlap, kinetic energy,
and nuclear attraction matrices — that are the building blocks of every electronic structure method.